In [1]:
# load the dataset cora
import torch
from torch_geometric.data import Data
from dataprocessing.partitioning import label_dirichlet_partition
from dataprocessing.datasets import GraphDataset
from dataprocessing.loaders import load_dataset, load_and_split, load_and_split_with_khop, load_and_split_with_feature_prop
import numpy as np
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


/home/brian_bosho/FP/FP/dataprocessing/data_utils.py:59: SyntaxWarning: invalid escape sequence '\m'
  """


### Base test functions

In [2]:
def print_subgraph_stats(data: Data, name: str = "Subgraph"):
    """
    Prints basic statistics for a PyTorch Geometric Data object.
    
    Args:
        data: PyTorch Geometric Data object
        name: Name/identifier for the subgraph (default="Subgraph")
    """
    print(f"{name} stats:")
    print(f"Number of nodes: {data.num_nodes}")
    print(f"Number of edges: {data.edge_index.size(1)}")
    print(f"Number of features: {data.x.size(1)}")
    print(f"Number of classes: {len(torch.unique(data.y))}")
    print(f"Number of training nodes: {data.train_mask.sum().item()}")
    print(f"Number of validation nodes: {data.val_mask.sum().item()}")
    print(f"Number of test nodes: {data.test_mask.sum().item()}")
    print(f"Zero feature vectors: {(data.x.sum(dim=1) == 0).sum().item()}")
    print("-------------------")

# Example usage:
# print_subgraph_stats(subgraph, "Original subgraph")
# print_subgraph_stats(expanded_subgraph, "Expanded subgraph")

In [3]:
def check_connectivity_to_original(new_subgraph: Data, mapping: torch.Tensor):
    """
    Checks if all nodes in the new subgraph are connected to the original nodes.

    Args:
        new_subgraph: The expanded subgraph Data object.
        mapping: Tensor mapping original node indices to new subgraph indices.

    Returns:
        A boolean indicating if all nodes are connected to the original nodes.
    """
    # Set of original node indices in the new subgraph
    original_node_indices = set(mapping.tolist())

    # Check connectivity for each node in the new subgraph
    for node in range(new_subgraph.num_nodes):
        # Get neighbors of the current node
        mask = (new_subgraph.edge_index[0] == node) | (new_subgraph.edge_index[1] == node)
        neighbors = set(new_subgraph.edge_index[:, mask].unique().tolist())

        # Check if any neighbor is an original node
        if not neighbors.intersection(original_node_indices):
            print(f"Node {node} is not connected to any original node.")
            return False

    print("All nodes in the new subgraph are connected to the original nodes.")
    return True



In [4]:
from torch_geometric.utils import k_hop_subgraph
def get_subgraph_1_hop(subgraph: Data, full_graph: Data, node_indices: torch.Tensor):
    """
    Computes the 1-hop subgraph of nodes specified by indices within the full graph.
    
    Args:
        subgraph: Original subgraph Data object
        full_graph: The complete graph Data object
        node_indices: Tensor of node indices in the full graph to get 1-hop neighbors for
    
    Returns:
        Data object containing the 1-hop subgraph
    """
    # Create boolean mask for the specified nodes
    subgraph_nodes_mask = torch.zeros(full_graph.num_nodes, dtype=torch.bool)
    subgraph_nodes_mask[node_indices] = True
    subgraph_nodes = torch.where(subgraph_nodes_mask)[0]

    # Get the 1-hop subgraph
    subset, edge_index, mapping, edge_mask = k_hop_subgraph(
        subgraph_nodes, 
        1, 
        full_graph.edge_index, 
        relabel_nodes=True
    )

    # Extract features, labels, and masks from the full graph using the subset
    x = full_graph.x[subset]
    y = full_graph.y[subset]
    train_mask = full_graph.train_mask[subset]
    val_mask = full_graph.val_mask[subset]
    test_mask = full_graph.test_mask[subset]

    new_subgraph = Data(x=x, edge_index=edge_index, y=y, 
                       train_mask=train_mask, val_mask=val_mask, test_mask=test_mask)

    return new_subgraph, mapping

In [5]:
def prepare_expanded_subgraph_for_propagation(original_subgraph: Data, expanded_subgraph: Data, original_indices: torch.Tensor):
    """
    Prepares expanded subgraph for feature propagation by:
    - Zeroing features of new nodes (non-original nodes)
    - Setting appropriate masks (only original nodes used for training)
    - Maintaining original features and labels for initial nodes
    """
    # Determine device from original subgraph
    device = original_subgraph.x.device
    
    # Get the mapping of original nodes in the expanded graph
    original_nodes_mask = torch.zeros(expanded_subgraph.num_nodes, dtype=torch.bool, device=device)
    
    # The k_hop_subgraph function returns nodes in order where original nodes come first
    # This is guaranteed by the relabel_nodes=True parameter
    original_nodes_mask[:len(original_indices)] = True
    
    # Print some verification info
    print(f"Original nodes: {len(original_indices)}")
    print(f"Expanded nodes: {expanded_subgraph.num_nodes}")
    print(f"Original nodes in expanded graph: {original_nodes_mask.sum().item()}")
    
    # Create new feature matrix (all zeros initially)
    new_x = torch.zeros_like(expanded_subgraph.x, device=device)
    # Copy original features for original nodes
    new_x[original_nodes_mask] = original_subgraph.x
    
    # Create new masks (only original nodes are used for training)
    new_train_mask = torch.zeros(expanded_subgraph.num_nodes, dtype=torch.bool, device=device)
    new_val_mask = torch.zeros(expanded_subgraph.num_nodes, dtype=torch.bool, device=device)
    new_test_mask = torch.zeros(expanded_subgraph.num_nodes, dtype=torch.bool, device=device)
    
    # Copy original masks for original nodes
    new_train_mask[original_nodes_mask] = original_subgraph.train_mask
    new_val_mask[original_nodes_mask] = original_subgraph.val_mask
    new_test_mask[original_nodes_mask] = original_subgraph.test_mask
    
    # Create new labels (zeros for new nodes)
    new_y = torch.zeros(expanded_subgraph.num_nodes, dtype=expanded_subgraph.y.dtype, device=device)
    new_y[original_nodes_mask] = original_subgraph.y
    
    # Create new Data object
    prepared_subgraph = Data(
        x=new_x,
        edge_index=expanded_subgraph.edge_index.to(device),  # Ensure edge_index is also on correct device
        y=new_y,
        train_mask=new_train_mask,
        val_mask=new_val_mask,
        test_mask=new_test_mask
    )
    
    return prepared_subgraph

In [6]:
def analyze_graph_connectivity(data):
    """
    Analyzes and prints connectivity statistics for a PyTorch Geometric Data object.
    
    Args:
        data: PyTorch Geometric Data object containing edge_index and number of nodes
    """
    import torch
    import networkx as nx
    from torch_geometric.utils import degree, to_networkx
    
    # 1. Calculate node degrees
    node_degrees = degree(data.edge_index[0], num_nodes=data.num_nodes)
    
    # 2. Find isolated nodes
    isolated_nodes = torch.where(node_degrees == 0)[0]
    print("=== Isolated Nodes Analysis ===")
    print(f"Number of isolated nodes: {len(isolated_nodes)}")
    # print(f"Isolated nodes: {isolated_nodes.tolist()}")
    
    # 3. Get degree distribution
    # print("\n=== Degree Distribution ===")
    # degree_counts = torch.bincount(node_degrees.long())
    # for deg, count in enumerate(degree_counts):
    #     if count > 0:
    #         print(f"Nodes with degree {deg}: {count}")
    
    # 4. Basic connectivity statistics
    print("\n=== Connectivity Statistics ===")
    # print(f"Total nodes: {data.num_nodes}")
    # print(f"Total edges: {data.edge_index.shape[1]}")
    print(f"Average degree: {node_degrees.float().mean():.2f}")
    print(f"Maximum degree: {node_degrees.max()}")
    print(f"Minimum degree: {node_degrees.min()}")
    
    # 5. Connected components analysis
    G = to_networkx(data, to_undirected=True)
    connected_components = list(nx.connected_components(G))
    print("\n=== Connected Components Analysis ===")
    print(f"Number of connected components: {len(connected_components)}")
    print(f"Sizes of connected components: {[len(c) for c in connected_components]}")
    
    # return {
    #     'isolated_nodes': isolated_nodes,
    #     'node_degrees': node_degrees,
    #     'connected_components': connected_components
    # }



### Print dataset stats

In [15]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [60]:
# load data with split
data, dataset, clients_data, test_data = load_and_split("Cora",device= device, num_clients=10, beta=0.5)
# print stats
# pick client 1 and get stats
client_1 = clients_data[1]
# print stats
print_subgraph_stats(client_1, "Client 1")

# print connectivity stats
analyze_graph_connectivity(client_1)


Client 1 stats:
Number of nodes: 367
Number of edges: 548
Number of features: 1433
Number of classes: 4
Number of training nodes: 18
Number of validation nodes: 62
Number of test nodes: 151
Zero feature vectors: 0
-------------------
=== Isolated Nodes Analysis ===
Number of isolated nodes: 122

=== Connectivity Statistics ===
Average degree: 1.49
Maximum degree: 11.0
Minimum degree: 0.0

=== Connected Components Analysis ===
Number of connected components: 147
Sizes of connected components: [47, 1, 113, 5, 1, 2, 2, 13, 2, 4, 9, 1, 1, 1, 1, 1, 5, 1, 1, 1, 2, 1, 1, 1, 1, 1, 2, 1, 9, 1, 2, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 2, 1, 3, 1, 1, 1, 1, 1, 1, 4, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 2, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1]


In [61]:
# load data with split
data, dataset, clients_data, test_data = load_and_split_with_khop("Cora",device= device, num_clients=10, beta=0.5, hop=1)
# print stats
# pick client 1 and get stats
client_1 = clients_data[1]
# print stats
print_subgraph_stats(client_1, "Client 1")

# print connectivity stats
analyze_graph_connectivity(client_1)

Client 1 stats:
Number of nodes: 918
Number of edges: 3378
Number of features: 1433
Number of classes: 4
Number of training nodes: 18
Number of validation nodes: 62
Number of test nodes: 151
Zero feature vectors: 551
-------------------
=== Isolated Nodes Analysis ===
Number of isolated nodes: 0

=== Connectivity Statistics ===
Average degree: 3.68
Maximum degree: 55.0
Minimum degree: 1.0

=== Connected Components Analysis ===
Number of connected components: 37
Sizes of connected components: [807, 5, 5, 2, 7, 2, 3, 2, 2, 12, 3, 2, 4, 6, 2, 3, 3, 2, 4, 2, 2, 4, 3, 2, 4, 2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 2, 2]


In [62]:
# load data with split
data, dataset, clients_data, test_data = load_and_split_with_feature_prop("Cora",device= device, num_clients=10, beta=0.5, hop=2)
# print stats
# pick client 1 and get stats
client_1 = clients_data[1]
# print stats
print_subgraph_stats(client_1, "Client 1")

# print connectivity stats
analyze_graph_connectivity(client_1)

Client 1 stats:
Number of nodes: 1768
Number of edges: 7196
Number of features: 1433
Number of classes: 4
Number of training nodes: 18
Number of validation nodes: 62
Number of test nodes: 151
Zero feature vectors: 25
-------------------
=== Isolated Nodes Analysis ===
Number of isolated nodes: 0

=== Connectivity Statistics ===
Average degree: 4.07
Maximum degree: 168.0
Minimum degree: 1.0

=== Connected Components Analysis ===
Number of connected components: 22
Sizes of connected components: [1704, 5, 2, 9, 2, 3, 5, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 2, 2, 2, 2]


### Test with base functions

In [63]:

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cpu')
from dataprocessing.partitioning import label_dirichlet_partition
from dataprocessing.partitioning import create_subgraph

In [67]:


dataset_name = "Cora"
num_clients = 10
beta = 0.5

data, dataset = load_dataset(dataset_name)

labels = data.y.cpu().numpy()
N = len(labels)
K = len(np.unique(labels))

split_data_indexes = label_dirichlet_partition(labels, N, K, num_clients, beta)
    
    # Create test data
initial_subgraphs = [create_subgraph(data, indices) for indices in split_data_indexes]

subgraph = initial_subgraphs[0]
full_graph = data
node_indices = split_data_indexes[0]

print(f"Subgraph nodes: {subgraph}")
print(f"Full graph nodes: {full_graph}")
new_subgraph, mapping = get_subgraph_1_hop(subgraph, full_graph, node_indices)

if new_subgraph is not None:
    print(f"New subgraph nodes: {new_subgraph}")


Subgraph nodes: Data(x=[103, 1433], edge_index=[2, 54], y=[103], train_mask=[103], val_mask=[103], test_mask=[103])
Full graph nodes: Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
New subgraph nodes: Data(x=[344, 1433], edge_index=[2, 1056], y=[344], train_mask=[344], val_mask=[344], test_mask=[344])


In [68]:
data_mat = subgraph.x
# print the shape of data_mat
print(f"Shape of data_mat: {data_mat.shape}")



Shape of data_mat: torch.Size([103, 1433])


In [69]:
# check the min & max node indices in the edge indices of subgraph and new_subgraph
print(f"Maximum node index in subgraph: {subgraph.edge_index.max()}")
print(f"Minimum node index in subgraph: {subgraph.edge_index.min()}")
print(f"Maximum node index in new_subgraph: {new_subgraph.edge_index.max()}")
print(f"Minimum node index in new_subgraph: {new_subgraph.edge_index.min()}")

# number of nodes in subgraph and new_subgraph
print(f"Number of nodes in subgraph: {subgraph.num_nodes}")
print(f"Number of nodes in new_subgraph: {new_subgraph.num_nodes}")



Maximum node index in subgraph: 96
Minimum node index in subgraph: 8
Maximum node index in new_subgraph: 343
Minimum node index in new_subgraph: 0
Number of nodes in subgraph: 103
Number of nodes in new_subgraph: 344


In [70]:
# check zero vectors in the subgraph & new_subgraph
zero_vectors = (subgraph.x.sum(dim=1) == 0).sum().item()
print(f"Number of zero vectors in subgraph: {zero_vectors}")
zero_vectors = (new_subgraph.x.sum(dim=1) == 0).sum().item()
print(f"Number of zero vectors in new_subgraph: {zero_vectors}")



Number of zero vectors in subgraph: 0
Number of zero vectors in new_subgraph: 0


In [ ]:
prepared_subgraph = prepare_expanded_subgraph_for_propagation(subgraph, new_subgraph, mapping)

# check the min & max node indices in the edge indices of subgraph and prepared_subgraph
print(f"Maximum node index in subgraph: {subgraph.edge_index.max()}")
print(f"Minimum node index in subgraph: {subgraph.edge_index.min()}")
print(f"Maximum node index in prepared_subgraph: {prepared_subgraph.edge_index.max()}")
print(f"Minimum node index in prepared_subgraph: {prepared_subgraph.edge_index.min()}")



In [ ]:
# Usage example:
results = analyze_graph_connectivity(subgraph)

In [ ]:
# Usage example:
results = analyze_graph_connectivity(new_subgraph)

In [ ]:
# check nodes with zero feature vectors in the prepared_subgraph
zero_vectors = (prepared_subgraph.x.sum(dim=1) == 0).sum().item()
print(f"Number of zero vectors in prepared_subgraph: {zero_vectors}")





#### Load data from codebase

In [16]:
# load and split with khop
kh_data, kh_dataset, kh_clients_data, kh_test_data = load_and_split_with_khop("Cora",device, num_clients=10, beta=0.5, hop=1)

In [17]:
kh_subgraph0 = kh_clients_data[0]

In [ ]:
kh_subgraph0

In [ ]:
# get the matrix of kh_subgraph0
kh_subgraph0_mat = kh_subgraph0.x
# print the shape of kh_subgraph0_mat
print(f"Shape of kh_subgraph0_mat: {kh_subgraph0_mat.shape}")



In [ ]:
# compare the shape of kh_subgraph0_mat and subgraph.x and new_subgraph.x and prepared_subgraph.x
print(f"Shape of subgraph.x: {subgraph.x.shape}")
print(f"Shape of kh_subgraph0_mat: {kh_subgraph0_mat.shape}")
print(f"Shape of new_subgraph.x: {new_subgraph.x.shape}")
print(f"Shape of prepared_subgraph.x: {prepared_subgraph.x.shape}")





In [ ]:
# check maximum and minimum node indices in kh_subgraph0_mat fromthe edge indices
print(f"Maximum node index in kh_subgraph0_mat: {kh_subgraph0.edge_index.max()}")
print(f"Minimum node index in kh_subgraph0_mat: {kh_subgraph0.edge_index.min()}")


In [ ]:
# check of prepared_subgraph.x is similar to kh_subgraph0_mat
are_similar = torch.allclose(prepared_subgraph.x, kh_subgraph0_mat)
print(f"Are prepared_subgraph.x and kh_subgraph0_mat similar? {are_similar}")

In [ ]:
# check if edge indices of kh_subgraph0 & prepared_subgraph are the same
print(f"Edge indices of kh_subgraph0: {kh_subgraph0.edge_index}")
print(f"Edge indices of prepared_subgraph: {prepared_subgraph.edge_index}")
# print their shapes
print(f"Shape of kh_subgraph0.edge_index: {kh_subgraph0.edge_index.shape}")
print(f"Shape of prepared_subgraph.edge_index: {prepared_subgraph.edge_index.shape}")

# check min & max node indices in kh_subgraph0 & prepared_subgraph edge indices
print(f"Minimum node index in kh_subgraph0.edge_index: {kh_subgraph0.edge_index.min()}")
print(f"Maximum node index in kh_subgraph0.edge_index: {kh_subgraph0.edge_index.max()}")
print(f"Minimum node index in prepared_subgraph.edge_index: {prepared_subgraph.edge_index.min()}")
print(f"Maximum node index in prepared_subgraph.edge_index: {prepared_subgraph.edge_index.max()}")





























In [ ]:
# check the number of nodes for training test & validation in kh_subgraph0 & prepared_subgraph & subgraph
print("Total number of nodes in kh_subgraph0: ", kh_subgraph0.num_nodes)
print(f"Number of nodes in kh_subgraph0: {kh_subgraph0.num_nodes}")
print(f"Number of nodes in prepared_subgraph: {prepared_subgraph.num_nodes}")
print(f"Number of nodes in subgraph: {subgraph.num_nodes}")

# check the number of training nodes for kh_subgraph0 & prepared_subgraph & subgraph
print(f"Number of training nodes in kh_subgraph0: {kh_subgraph0.train_mask.sum()}")
print(f"Number of training nodes in prepared_subgraph: {prepared_subgraph.train_mask.sum()}")
print(f"Number of training nodes in subgraph: {subgraph.train_mask.sum()}")

# check the number of test nodes for kh_subgraph0 & prepared_subgraph & subgraph
print(f"Number of test nodes in kh_subgraph0: {kh_subgraph0.test_mask.sum()}")
print(f"Number of test nodes in prepared_subgraph: {prepared_subgraph.test_mask.sum()}")
print(f"Number of test nodes in subgraph: {subgraph.test_mask.sum()}")

In [26]:
from tabulate import tabulate

def display_graph_stats(graph, name="Graph"):
    """
    Displays graph statistics in a formatted table.
    
    Args:
        graph: PyTorch Geometric Data object
        name: Name of the graph for display purposes
    """
    # Move tensors to CPU for safe computation
    edge_index_cpu = graph.edge_index.cpu()
    
    # Collect statistics
    stats = [
        [name, "Value"],
        ["Total Nodes", graph.num_nodes],
        ["Total Edges", edge_index_cpu.shape[1]],
        ["Maximum Node Index", edge_index_cpu.max().item()],
        ["Minimum Node Index", edge_index_cpu.min().item()],
        ["Training Nodes", graph.train_mask.sum().item()],
        ["Validation Nodes", graph.val_mask.sum().item()],
        ["Test Nodes", graph.test_mask.sum().item()]
    ]
    
    # Create and display table
    print(tabulate(stats, headers="firstrow", tablefmt="grid"))

# Example usage:
# display_graph_stats(kh_subgraph0, "KH Subgraph")
# display_graph_stats(prepared_subgraph, "Prepared Subgraph")
# display_graph_stats(subgraph, "Subgraph")

# To compare multiple graphs side by side:
def compare_graphs(*graphs, names=None):
    """
    Displays statistics for multiple graphs side by side in a table.
    
    Args:
        *graphs: Variable number of PyTorch Geometric Data objects
        names: List of names for the graphs (optional)
    """
    if names is None:
        names = [f"Graph {i+1}" for i in range(len(graphs))]
    
    # Prepare headers
    headers = ["Metric"] + names
    
    # Prepare rows
    metrics = [
        "Total Nodes",
        "Total Edges",
        "Max Node Index",
        "Min Node Index",
        "Training Nodes",
        "Validation Nodes",
        "Test Nodes"
    ]
    
    rows = []
    for metric in metrics:
        row = [metric]
        for graph in graphs:
            if metric == "Total Nodes":
                value = graph.num_nodes
            elif metric == "Total Edges":
                value = graph.edge_index.shape[1]
            elif metric == "Max Node Index":
                value = graph.edge_index.cpu().max().item()
            elif metric == "Min Node Index":
                value = graph.edge_index.cpu().min().item()
            elif metric == "Training Nodes":
                value = graph.train_mask.sum().item()
            elif metric == "Validation Nodes":
                value = graph.val_mask.sum().item()
            elif metric == "Test Nodes":
                value = graph.test_mask.sum().item()
            row.append(value)
        rows.append(row)
    
    print(tabulate(rows, headers=headers, tablefmt="grid"))

# Example usage:


In [ ]:
compare_graphs(
    kh_subgraph0, 
    prepared_subgraph, 
    new_subgraph,
    subgraph, 
    names=["KH Subgraph", "Prepared Subgraph", "New Subgraph", "Subgraph"]
) 

### More stuff

In [ ]:
#chek the maximum and minium node indices in the edge indices of subgraph and prepared_subgraph
print(f"Maximum node index in subgraph: {subgraph.edge_index.max()}")
print(f"Minimum node index in subgraph: {subgraph.edge_index.min()}")
print(f"Maximum node index in prepared_subgraph: {prepared_subgraph.edge_index.max()}")
print(f"Minimum node index in prepared_subgraph: {prepared_subgraph.edge_index.min()}")
# do the same for new_subgraph
print(f"Maximum node index in new_subgraph: {new_subgraph.edge_index.max()}")
print(f"Minimum node index in new_subgraph: {new_subgraph.edge_index.min()}")


In [ ]:
# # Example usage with verification
# expanded_subgraph = get_subgraph_1_hop(subgraph, full_graph, original_indices)
# prepared_subgraph = prepare_expanded_subgraph_for_propagation(subgraph, expanded_subgraph, original_indices)

# Verify
print(f"Original features sum: {subgraph.x.sum()}")
print(f"New features sum: {prepared_subgraph.x.sum()}")  # Should be same as original
print(f"Training nodes: {prepared_subgraph.train_mask.sum()}")  # Should match original

In [ ]:
# print statistics for original
print_subgraph_stats(subgraph, "Original subgraph")
print_subgraph_stats(prepared_subgraph, "Prepared subgraph")

In [272]:
# now lets do feature propagation on the prepared subgraph
from dataprocessing.data_utils import propagate_features

# get nonzero feature vector mask, which is the mapping of the original nodes
zero_vector_mask = (prepared_subgraph.x == 0).all(dim=1)

In [ ]:
len(zero_vector_mask)
# number of zero vectors
print(zero_vector_mask.sum())


In [ ]:


non_zero_vector_mask = ~zero_vector_mask

propagated_subgraph = propagate_features(prepared_subgraph.x, prepared_subgraph.edge_index, non_zero_vector_mask)
print(propagated_subgraph)


In [275]:
propagated_features = propagated_subgraph

In [ ]:
# 1. Check if they're exactly equal
are_equal = torch.equal(propagated_subgraph, prepared_subgraph.x)
print("Are tensors exactly equal?", are_equal)


In [ ]:
# 2. Check the difference in values
difference = torch.abs(propagated_subgraph - prepared_subgraph.x)
print("Maximum difference:", difference.max().item())
print("Average difference:", difference.mean().item())


In [ ]:
num_changed = (propagated_features != prepared_subgraph.x).sum().item()
print(f"Number of changed elements: {num_changed} out of {propagated_features.numel()}")


In [ ]:
print("Original features sum:", prepared_subgraph.x.sum().item())
print("Propagated features sum:", propagated_features.sum().item())


In [ ]:
#  5. Check if any zero vectors became non-zero
original_zero_vectors = (prepared_subgraph.x.sum(dim=1) == 0).sum().item()
propagated_zero_vectors = (propagated_features.sum(dim=1) == 0).sum().item()
print(f"Zero vectors before: {original_zero_vectors}")
print(f"Zero vectors after: {propagated_zero_vectors}")

In [281]:
# load dataset with feature propagation
data, dataset, clients_data, test_data = load_and_split_with_feature_prop("Cora", num_clients=10, beta=0.5, hop=1)

In [284]:
features2 = clients_data[1].x

In [ ]:
# coun zero vectors in features2
zero_vectors = (features2.sum(dim=1) == 0).sum().item()
print(f"Number of zero vectors in features2: {zero_vectors}")
# count non zero vectors in features2
non_zero_vectors = (features2.sum(dim=1) != 0).sum().item()
print(f"Number of non zero vectors in features2: {non_zero_vectors}")



In [ ]:
# check if features2 is similar to propagated_features
are_similar = torch.allclose(features2, propagated_features)
print(f"Are features2 and propagated_features similar? {are_similar}")

# check the difference between features2 and propagated_features
difference = torch.abs(features2 - propagated_features)
print(f"Difference between features2 and propagated_features: {difference.max().item()}")

### Test the functions 

In [7]:
data, dataset, clients_data, test_data = load_and_split("Cora", device, num_clients=10, beta=0.5)
subgraph = clients_data[0]

# print statistics for original
print_subgraph_stats(subgraph, "Load and split")


Load and split stats:
Number of nodes: 103
Number of edges: 54
Number of features: 1433
Number of classes: 5
Number of training nodes: 11
Number of validation nodes: 16
Number of test nodes: 34
Zero feature vectors: 0
-------------------


In [8]:
test_data[0]
print_subgraph_stats(test_data[0], "Test data")

Test data stats:
Number of nodes: 103
Number of edges: 54
Number of features: 1433
Number of classes: 5
Number of training nodes: 11
Number of validation nodes: 16
Number of test nodes: 34
Zero feature vectors: 0
-------------------


In [ ]:
# test load and split with khop
data, dataset, clients_data, test_data  = load_and_split_with_khop("Citeseer", device, num_clients=10, beta=0.5, hop=2,  fulltraining_flag=True)


print_subgraph_stats(clients_data[0], "Load and split with khop")
print_subgraph_stats(test_data[0], "Test data")

TypeError: load_and_split_with_khop() got an unexpected keyword argument 'use_feature_prop'

In [12]:
print_subgraph_stats(clients_data[1], "Load and split with khop")
print_subgraph_stats(test_data[1], "Test data")

Load and split with khop stats:
Number of nodes: 1671
Number of edges: 5608
Number of features: 3703
Number of classes: 6
Number of training nodes: 58
Number of validation nodes: 264
Number of test nodes: 501
Zero feature vectors: 127
-------------------
Test data stats:
Number of nodes: 332
Number of edges: 126
Number of features: 3703
Number of classes: 6
Number of training nodes: 10
Number of validation nodes: 46
Number of test nodes: 85
Zero feature vectors: 1
-------------------


In [13]:
data

Data(x=[3327, 3703], edge_index=[2, 9104], y=[3327], train_mask=[3327], val_mask=[3327], test_mask=[3327])